# Figures for *Estimating Neural Timescales*

Every figure in the paper is drawn here, one figure per cell, with its complete
matplotlib code, so any of them can be edited and re-run on its own.

* **Run the setup cell first.** It sets paths, the shared style, the shared
  colours and the constructed AR(7) used by two figures.
* Each figure cell reads the cached tables in `results/` (written by the fit
  scripts, `scripts/run_0*.py`) and writes `figures/<name>.svg` + `.png`.
  Nothing here refits the expensive simulations; the only fitting is the
  handful of example fits a panel needs, so every cell runs in seconds
  (`fig_realdata` takes about a minute, since it fits every trace of every
  recording).
* `scripts/run_notebook.py notebooks/figures.ipynb` executes every cell in
  order without Jupyter, and is what the DVC `plot_figures` stage runs.
* `scripts/sync_overleaf.py` copies the figures into the Overleaf project.

| Paper | File | Reads |
|---|---|---|
| Fig. 1 | `fig_ar1.svg` | simulation in the cell |
| Fig. 2 | `fig_bias.svg` | simulation in the cell |
| Fig. 3 | `fig_modes.svg` | `modes_vs_coefficients_repeatability.csv` |
| Fig. 4 | `fig_oscillations.svg` | `oscillation_sweep.csv.gz`, `damped_grid.csv.gz` |
| Fig. 5 | `fig_finite.svg` | `finite_sampling.csv.gz` + simulation in the cell |
| Fig. 6 | `fig_realdata.svg` | `real_data_fits.csv`, `data/*.npz` |
| Fig. S1 | `figS_benchmark.svg` | `benchmark_main.csv.gz` |
| Fig. S2 | `figS_oscillation_grid.svg` | `oscillation_sweep.csv.gz` |
| Fig. S3 | `figS_windows.svg` | `empirical_truncation_summary.csv` |

In [ ]:
# ---- Setup: run first ------------------------------------------------------
import os, sys, json
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '1')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# Works whether the kernel starts in revision/ or revision/notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'tsmodes').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
FIG, RES = ROOT / 'figures', ROOT / 'results'
FIG.mkdir(exist_ok=True)

from tsmodes import plotting as P
from tsmodes import datasets as D
from tsmodes.estimators import ESTIMATOR_LABELS, estimate, prepare
from tsmodes.modes import (ARModes, ar_acf, ar_acf_fft, ar_psd, poles_to_ar,
                           psd_from_poles)
from tsmodes.sims import (AR1, AR1PlusSine, AR2Damped, ARpModes, PowerLaw,
                          sim_ar_process)
from tsmodes.finite import phi_of_tau, tau_of_phi

P.use_style()
FS = 500.0            # sampling rate of every simulation
TAU = 0.05            # the constructed timescale of the AR(1) simulations, s

# Colours shared by every figure.  Data is black and solid; an AR fit is this
# blue and dashed; modes are coloured by importance rank.  The two mode-kind
# colours (relaxation blue, oscillatory red) live in tsmodes.plotting.
DATA_COLOR = '#000000'
FIT_COLOR = '#0072B2'
FIT_STYLE = (0, (4, 2))
FIT_LW = 1.5
RANK_COLORS = ['#D55E00', '#009E73', '#CC79A7']

# The constructed AR(7) read mode by mode in fig_modes and used as the
# model-mismatch column of fig_bias: two relaxations of comparable importance
# an order of magnitude apart, two rhythms that ring for more than a cycle,
# and one fast relaxation that does not matter.
TRUE_MODES = [
    {'kind': 'relaxation', 'tau': 0.200, 'freq': 0.0},
    {'kind': 'oscillatory', 'tau': 0.150, 'freq': 10.0},
    {'kind': 'oscillatory', 'tau': 0.060, 'freq': 40.0},
    {'kind': 'relaxation', 'tau': 0.020, 'freq': 0.0},
    {'kind': 'relaxation', 'tau': 0.002, 'freq': 0.0},
]


def build_poles(spec, fs):
    poles = []
    for m in spec:
        r = np.exp(-1.0 / (m['tau'] * fs))
        if m['kind'] == 'relaxation':
            poles.append(r + 0j)
        else:
            theta = 2 * np.pi * m['freq'] / fs
            poles += [r * np.exp(1j * theta), r * np.exp(-1j * theta)]
    return np.array(poles)


PHI7 = poles_to_ar(build_poles(TRUE_MODES, FS))


def proxy(**kw):
    """A legend handle that is not tied to a drawn line."""
    return Line2D([0], [0], **kw)


def save(fig, name):
    """Write SVG and PNG into figures/, and leave the figure open so Jupyter
    shows it under the cell."""
    fig.savefig(FIG / name, format='svg')
    fig.savefig(FIG / name.replace('.svg', '.png'), format='png')
    print('wrote', FIG / name)
    return fig


def decade_ticks(ax, vals, pad=1.7):
    """Two or three labelled ticks on a log axis, whatever the span."""
    lo = float(np.min(vals)) / pad
    hi = float(np.max(vals)) * pad
    ax.set_ylim(lo, hi)
    best = None
    for mant in ([1.0], [1.0, 3.0], [1.0, 2.0, 5.0],
                 [1.0, 1.5, 2.0, 3.0, 5.0, 7.0]):
        ticks = np.array(sorted(m * 10.0 ** e
                                for e in range(-9, 10) for m in mant))
        ticks = ticks[(ticks >= lo) & (ticks <= hi)]
        if len(ticks) > 3:
            ticks = ticks[::int(np.ceil(len(ticks) / 3.0))]
        if best is None or len(ticks) > len(best):
            best = ticks
        if len(best) >= 3:
            break
    if best is not None and len(best) >= 2:
        ax.set_yticks(best)
        ax.set_yticklabels([f'{t:g}' for t in best])
    ax.yaxis.set_minor_locator(mticker.NullLocator())


print('root:', ROOT)

## Fig. 1 — three views of the same model (`fig_ar1.svg`)

An AR(1) process as a signal, an autocorrelation and a power spectrum, with the
AR(1) form of each written beneath it. Replaces the hand-drawn `fig_0.svg`.

In [ ]:
# ---- Fig. 1: fig_ar1 -------------------------------------------------------
rng = np.random.default_rng(0)
proc = AR1(tau=TAU)
sig = proc.simulate(60.0, FS, rng)
signal = prepare(sig, FS, welch_win_s=4.0, nlags=200)
phi = phi_of_tau(TAU, FS)

fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.5),
                         gridspec_kw=dict(wspace=0.45))
EQ_Y = -0.38          # equation baseline, in axes fraction

# -- a: the signal -----------------------------------------------------------
ax = axes[0]
t = np.arange(int(1.0 * FS)) / FS
ax.plot(t, sig[:len(t)] / np.std(sig), color=DATA_COLOR, lw=0.7)
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.set_xlabel('Time (s)')
ax.set_title('Signal', fontsize=9)
ax.text(0.5, EQ_Y, r'$x_t = \varphi\, x_{t-1} + \epsilon$',
        transform=ax.transAxes, ha='center', va='top', fontsize=8.5)
P.panel_label(ax, 'a', dx=-0.10)

# -- b: the autocorrelation --------------------------------------------------
ax = axes[1]
nl = 100
lags = np.arange(nl + 1) / FS
ax.plot(lags, signal.corrs[:nl + 1], color=DATA_COLOR, lw=1.2, label='data')
ax.plot(lags, phi ** np.arange(nl + 1), color=FIT_COLOR, lw=FIT_LW,
        ls=FIT_STYLE, label='AR(1)')
ax.plot([1 / FS], [phi], 'o', color=FIT_COLOR, ms=4)
ax.annotate(r'$A(1) = \varphi$', xy=(1 / FS, phi), xytext=(0.06, 0.82),
            fontsize=7, arrowprops=dict(arrowstyle='->', lw=0.7, color=P.GREY))
ax.axhline(0, color=P.LIGHT, lw=0.6, zorder=0)
ax.set_xlabel('Lag (s)')
ax.set_ylabel('Autocorrelation')
ax.set_title('Autocorrelation', fontsize=9)
ax.legend(fontsize=6.5, loc='upper right', handlelength=1.8)
ax.text(0.5, EQ_Y, r'$A(k) = \varphi^{k} = e^{-k/(\tau f_s)}$',
        transform=ax.transAxes, ha='center', va='top', fontsize=8.5)
P.panel_label(ax, 'b', dx=-0.28)

# -- c: the power spectrum ---------------------------------------------------
ax = axes[2]
mask = signal.fit_mask
f, p = signal.freqs[mask], signal.powers[mask]
theory = proc.psd(f, FS)
theory = theory * np.exp(np.mean(np.log(p)) - np.mean(np.log(theory)))
ax.loglog(f, p, color=DATA_COLOR, lw=1.0, label='data')
ax.loglog(f, theory, color=FIT_COLOR, lw=FIT_LW, ls=FIT_STYLE, label='AR(1)')
knee = 1 / (2 * np.pi * TAU)
ax.axvline(knee, color=P.LIGHT, lw=0.8, zorder=0)
ax.text(knee * 1.15, 0.92, r'$1/(2\pi\tau)$', transform=ax.get_xaxis_transform(),
        fontsize=7, color=P.GREY)
ax.set_yticks([])
ax.yaxis.set_minor_locator(mticker.NullLocator())
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power')
ax.set_title('Power spectrum', fontsize=9)
ax.legend(fontsize=6.5, loc='lower left', handlelength=1.8)
ax.text(0.5, EQ_Y, r'$P(f) = \sigma^2 \,/\, |1 - \varphi\, e^{-j 2\pi f / f_s}|^{2}$',
        transform=ax.transAxes, ha='center', va='top', fontsize=8.5)
P.panel_label(ax, 'c', dx=-0.22)

save(fig, 'fig_ar1.svg')

## Fig. 2 — four ways a timescale estimate goes wrong (`fig_bias.svg`)

Columns: model mismatch (the AR(7) of Fig. 3), oscillations, finite samples,
dynamic timescales. Rows: signal, autocorrelation, power spectrum, each with
the AR(1) fit overlaid. In the dynamic column the AR(1) coefficient rises over
the recording; evenly spaced windows are boxed in colour and the theoretical
AR(1) curve for each window is drawn in its colour. Replaces the hand-drawn
`fig_1.svg`.

In [ ]:
# ---- Fig. 2: fig_bias ------------------------------------------------------
# The estimator drawn in every column is the one the field uses: an
# exponential, A phi^k + B, fit to the sample autocorrelation.  Its Lorentzian
# is drawn over the spectrum so both rows show the same AR(1).
from scipy.optimize import curve_fit

rng = np.random.default_rng(1)
NL = 150                                  # lags drawn
ks = np.arange(NL + 1)
lags = ks / FS


def sample_acf(x, nlags=NL):
    """Plain sample autocorrelation, normalized to one at lag zero."""
    x = x - x.mean()
    full = np.correlate(x, x, mode='full')[len(x) - 1:]
    return full[:nlags + 1] / full[0]


def fit_exponential(acf):
    """A * phi^k + B, the standard exponential fit to an autocorrelation."""
    model = lambda k, A, phi, B: A * phi ** k + B
    (A, phi, B), _ = curve_fit(model, ks[:len(acf)], acf, p0=(1.0, 0.95, 0.0),
                               bounds=([0.0, 1e-6, -1.0], [2.0, 1 - 1e-9, 1.0]))
    return A, phi, B


def spectrum(sig, welch_win_s):
    signal = prepare(sig, FS, welch_win_s=welch_win_s)
    m = signal.fit_mask
    return signal.freqs[m], signal.powers[m]


def column(sig, welch_win_s=4.0):
    acf = sample_acf(sig)
    A, phi, B = fit_exponential(acf)
    f, p = spectrum(sig, welch_win_s)
    return dict(acf=acf, A=A, phi=phi, B=B, f=f, p=p)


def centred(power, ref):
    """Scale a theoretical spectrum onto the data's geometric mean."""
    return power * np.exp(np.mean(np.log(ref)) - np.mean(np.log(power)))


# a) model mismatch: the constructed AR(7)
sig_a = sim_ar_process(PHI7, int(60 * FS), rng)
col_a = column(sig_a)

# b) oscillations: an AR(1) with a 10 Hz sinusoid added
sig_b = AR1PlusSine(tau=TAU, f_osc=10.0, snr=1.0).simulate(60.0, FS, rng)
col_b = column(sig_b)

# c) finite samples: one second of an AR(1), twenty timescales of data
sig_c = AR1(tau=TAU).simulate(1.0, FS, rng)
col_c = column(sig_c, welch_win_s=0.5)

# d) dynamic timescales: an AR(1) whose coefficient rises over the recording
T_D = 8.0
n_d = int(T_D * FS)
tau_t = np.linspace(0.010, 0.200, n_d)             # 10 ms -> 200 ms
phi_t = phi_of_tau(tau_t, FS)
eps = rng.normal(size=n_d)
sig_d = np.zeros(n_d)
for i in range(1, n_d):
    sig_d[i] = phi_t[i] * sig_d[i - 1] + eps[i]
col_d = column(sig_d, welch_win_s=2.0)
WINDOWS = [(0.5, 1.5), (2.5, 3.5), (4.5, 5.5), (6.5, 7.5)]   # s, evenly spaced
WIN_COLORS = plt.cm.viridis(np.linspace(0.05, 0.85, len(WINDOWS)))
win_tau = [float(np.mean(tau_t[int(a * FS):int(b * FS)])) for a, b in WINDOWS]

columns = [('Model mismatch', sig_a, col_a),
           ('Oscillations', sig_b, col_b),
           ('Finite samples', sig_c, col_c),
           ('Dynamic timescales', sig_d, col_d)]

fig = plt.figure(figsize=(7.4, 5.2))
gs = fig.add_gridspec(3, 4, hspace=0.62, wspace=0.40,
                      height_ratios=[0.75, 1.0, 1.0])

for j, (title, sig, c) in enumerate(columns):
    dynamic, finite = j == 3, j == 2

    # -- row 1: the signal ---------------------------------------------------
    ax = fig.add_subplot(gs[0, j])
    if dynamic:
        t = np.arange(len(sig)) / FS
        ax.plot(t, sig / np.std(sig), color=DATA_COLOR, lw=0.35)
        for (a, b), wc in zip(WINDOWS, WIN_COLORS):
            ax.axvspan(a, b, color=wc, alpha=0.28, lw=0)
    else:
        n_show = min(len(sig), int(1.5 * FS))
        t = np.arange(n_show) / FS
        ax.plot(t, sig[:n_show] / np.std(sig[:n_show]), color=DATA_COLOR, lw=0.5)
    ax.set_ylim(-4, 4)
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Time (s)', fontsize=7.5)
    ax.tick_params(labelsize=7)
    if j == 0:
        ax.set_ylabel('Signal', fontsize=8)
    P.panel_label(ax, 'abcd'[j], dx=-0.14, dy=1.42)

    # -- row 2: the autocorrelation and its exponential fit -----------------
    ax = fig.add_subplot(gs[1, j])
    ax.plot(lags[:len(c['acf'])], c['acf'], color=DATA_COLOR, lw=1.1,
            label='data', zorder=2)
    if dynamic:
        for tw, wc in zip(win_tau, WIN_COLORS):
            ax.plot(lags, phi_of_tau(tw, FS) ** ks, color=wc, lw=1.2,
                    label=f'$\\tau$ = {tw * 1e3:.0f} ms', zorder=1)
    if finite:
        ax.plot(lags, phi_of_tau(TAU, FS) ** ks, color=P.GREY, lw=1.0,
                ls=':', label='true', zorder=1)
    ax.plot(lags, c['A'] * c['phi'] ** ks + c['B'], color=FIT_COLOR, lw=FIT_LW,
            ls=FIT_STYLE, label='AR(1) fit', zorder=3)
    ax.axhline(0, color=P.LIGHT, lw=0.6, zorder=0)
    ax.set_ylim(-0.55, 1.05)
    ax.set_yticks([0, 1])
    ax.set_xlabel('Lag (s)', fontsize=7.5)
    ax.tick_params(labelsize=7)
    if j == 0:
        ax.set_ylabel('Autocorrelation', fontsize=8)
    if j == 0 or finite:
        ax.legend(fontsize=6, loc='upper right', handlelength=1.6)
    if dynamic:
        ax.legend(fontsize=5.4, loc='upper right', handlelength=1.4,
                  labelspacing=0.2)

    # -- row 3: the spectrum and the same fit's Lorentzian -------------------
    ax = fig.add_subplot(gs[2, j])
    f, p = c['f'], c['p']
    ax.loglog(f, p, color=DATA_COLOR, lw=1.0, zorder=2)
    if dynamic:
        for tw, wc in zip(win_tau, WIN_COLORS):
            ax.loglog(f, centred(ar_psd([phi_of_tau(tw, FS)], f, FS, 1.0), p),
                      color=wc, lw=1.2, zorder=1)
    if finite:
        ax.loglog(f, centred(ar_psd([phi_of_tau(TAU, FS)], f, FS, 1.0), p),
                  color=P.GREY, lw=1.0, ls=':', zorder=1)
    ax.loglog(f, centred(ar_psd([c['phi']], f, FS, 1.0), p), color=FIT_COLOR,
              lw=FIT_LW, ls=FIT_STYLE, zorder=3)
    ax.set_yticks([])
    ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.set_xlabel('Frequency (Hz)', fontsize=7.5)
    ax.tick_params(labelsize=7)
    if j == 0:
        ax.set_ylabel('Power', fontsize=8)

save(fig, 'fig_bias.svg')

## Fig. 3 — an AR($p$) read as modes (`fig_modes.svg`)

Reads `results/modes_vs_coefficients_repeatability.csv` (written by
`scripts/run_01_modes.py`) for panel e; everything else is computed from the
constructed model in the cell.

In [ ]:
# ---- Fig. 3: fig_modes -----------------------------------------------------
model = ARModes.from_phi(PHI7, FS, offset=1.0, freqs=np.linspace(0, FS / 2, 1025))
dom = model.dominant_subset(tol=0.1)
repeat = pd.read_csv(RES / 'modes_vs_coefficients_repeatability.csv')

RECOMBINED_COLOR = '#FFB000'     # product of modes (b) and sum of modes (c)
DOM_STYLE = (0, (2, 1.5))        # a dominant mode
NEG_STYLE = ':'                  # a negligible mode

fig = plt.figure(figsize=(7.4, 5.0))
# Bottom-row spans land on top-row panel edges: d under a, e under b and c.
gs = fig.add_gridspec(2, 6, hspace=0.62, wspace=1.05)
freqs = np.logspace(np.log10(0.5), np.log10(150), 600)
order = np.argsort([-m.tau for m in model.modes])       # slowest first
shown = [model.modes[i] for i in order]
colours = [P.MODE_COLORS[m.kind] for m in shown]
nlags = 150
lags = np.arange(nlags + 1) / model.fs
keep = set(dom['indices'])

style_handles = [
    proxy(color=P.MODE_COLORS['relaxation'], lw=1.4, label='relaxation mode'),
    proxy(color=P.MODE_COLORS['oscillatory'], lw=1.4, label='oscillatory mode'),
]
weight_handles = [
    proxy(color=P.GREY, lw=1.1, ls=DOM_STYLE, label='mode, dominant'),
    proxy(color=P.GREY, lw=0.9, ls=NEG_STYLE, alpha=0.6, label='mode, negligible'),
]
LEG = dict(frameon=True, framealpha=1.0, edgecolor='none', borderpad=0.22,
           labelspacing=0.2, handlelength=1.5)

# (a) the modes themselves ----------------------------------------------------
ax = fig.add_subplot(gs[0, 0:2])
n, step = 125, 1.35
t = np.arange(n) / model.fs
comps = model.impulse_components(n)
total = comps.sum(axis=0)
top = len(shown) * step
ax.plot(t, 0.5 * total / np.max(np.abs(total)) + top, color='k', lw=1.3)
ax.text(t[-1], top + 0.6, 'response to an input', fontsize=6.5, ha='right', va='bottom')
for i, mode in enumerate(shown):
    base = (len(shown) - 1 - i) * step
    y = 0.5 * comps[order[i]] / np.max(np.abs(comps[order[i]])) + base
    ax.axhline(base, color=P.LIGHT, lw=0.5, zorder=0)
    ax.plot(t, y, color=colours[i], lw=1.2)
    label = (f'$\\tau$ = {mode.tau * 1e3:.0f} ms'
             + ('' if mode.kind == 'relaxation' else f', {mode.freq:.0f} Hz'))
    ax.text(t[-1], base + 0.6, label, fontsize=6.2, ha='right', va='bottom',
            color=colours[i])
ax.set_yticks([])
ax.set_ylim(-1.35, top + 1.15)
ax.set_xlabel('Time after input (s)')
ax.set_title('Modes', fontsize=9)
ax.spines['left'].set_visible(False)
ax.legend(handles=style_handles, fontsize=5.8, loc='lower center', ncol=2,
          columnspacing=1.0, **LEG)
P.panel_label(ax, 'a', dx=-0.09)

# (b) the modes multiply in the spectrum --------------------------------------
ax = fig.add_subplot(gs[0, 2:4])
centre = lambda power: power / np.exp(np.mean(np.log(power)))
for j, (mode, c) in enumerate(zip(shown, colours)):
    dominant = order[j] in keep
    ax.loglog(freqs, centre(mode.psd(freqs)), color=c, zorder=1,
              lw=1.1 if dominant else 0.9, ls=DOM_STYLE if dominant else NEG_STYLE,
              alpha=1.0 if dominant else 0.6)
product = psd_from_poles(np.concatenate([m.poles for m in model.modes]),
                         freqs, model.fs, model.offset)
ax.loglog(freqs, centre(model.psd(freqs)), color='k', lw=1.6, zorder=3)
ax.loglog(freqs, centre(product), color=RECOMBINED_COLOR, lw=1.1, ls='--', zorder=4)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Relative power')
ax.set_title('Spectrum: modes multiply', fontsize=9)
ax.legend(handles=[proxy(color='k', lw=1.6, label='generating spectrum'),
                   proxy(color=RECOMBINED_COLOR, lw=1.1, ls='--', label='product of modes')]
          + weight_handles, fontsize=5.5, loc='lower left', **LEG)
P.panel_label(ax, 'b', dx=-0.26)

# (c) the modes add in the autocorrelation ------------------------------------
ax = fig.add_subplot(gs[0, 4:6])
acf_comps = model.acf_components(nlags)
for i, c in enumerate(colours):
    dominant = order[i] in keep
    ax.plot(lags, acf_comps[order[i]], color=c,
            lw=1.1 if dominant else 0.9, ls=DOM_STYLE if dominant else NEG_STYLE,
            alpha=1.0 if dominant else 0.6)
ax.plot(lags, model.acf(nlags), color='k', lw=1.6)
ax.plot(lags, acf_comps.sum(axis=0), color=RECOMBINED_COLOR, lw=1.1, ls='--')
ax.axhline(0, color=P.LIGHT, lw=0.6, zorder=0)
ax.set_xlim(0, 0.3)
ax.set_xlabel('Lag (s)')
ax.set_ylabel('Autocorrelation')
ax.set_title('Autocorrelation: modes add', fontsize=9)
ax.legend(handles=[proxy(color='k', lw=1.6, label='generating ACF'),
                   proxy(color=RECOMBINED_COLOR, lw=1.1, ls='--', label='sum of modes')]
          + weight_handles, fontsize=5.5, loc='upper right', **LEG)
P.panel_label(ax, 'c', dx=-0.26)

# (d) how many modes are needed ----------------------------------------------
ax = fig.add_subplot(gs[1, 0:2])
curve = dom['r2_curve']
ax.plot(np.arange(len(curve)), curve, 'o-', color='k', ms=4)
ax.axhline(0.99, color=P.GREY, ls='--', lw=1.0)
ax.text(0.1, 1.03, '$R^2 = 0.99$', fontsize=6.2, color=P.GREY, va='bottom')
ax.axvline(dom['n_modes'], color=P.LIGHT, lw=1.0, zorder=0)
ax.text(dom['n_modes'] + 0.15, 0.22, f"{dom['n_modes']} of {len(shown)}\nsuffice", fontsize=6.5)
ax.set_xlabel('Modes kept')
ax.set_ylabel('Spectrum reproduced ($R^2$)', fontsize=7.5)
ax.set_xticks(np.arange(len(curve)))
ax.set_ylim(-0.08, 1.18)
ax.set_title('Mode ranking', fontsize=9)
ax.text(0.97, 0.06, 'modes added in order\nof importance', transform=ax.transAxes,
        fontsize=5.8, color=P.GREY, ha='right', va='bottom')
P.panel_label(ax, 'd', dx=-0.34)

# (e) how well each mode's timescale is recovered ----------------------------
ax = fig.add_subplot(gs[1, 2:6])
idx = sorted(range(len(TRUE_MODES)), key=lambda i: -TRUE_MODES[i]['tau'])
vals, labels, bar_colors = [], [], []
for i in idx:
    spec = TRUE_MODES[i]
    row = repeat[repeat['quantity'] == f'mode{i}_tau'].iloc[0]
    vals.append(float(row['rel_error']))
    bar_colors.append(P.MODE_COLORS[spec['kind']])
    labels.append(f"{spec['tau'] * 1e3:g} ms"
                  + ('' if spec['kind'] == 'relaxation' else f", {spec['freq']:g} Hz"))
xs = np.arange(len(idx))
ax.bar(xs, vals, color=bar_colors, width=0.62, edgecolor='white')
for x, v in zip(xs, vals):
    ax.text(x, v * 1.25, f'{v * 100:.0f}%', ha='center', va='bottom', fontsize=6)
ax.set_yscale('log')
ax.set_ylim(min(vals) / 5.0, max(vals) * 12.0)
ax.set_xticks(xs)
ax.set_xticklabels(labels, fontsize=6.2, rotation=22, ha='right')
ax.set_xlabel('Constructed mode', fontsize=7.5)
ax.set_ylabel('Median error in $\\tau$', fontsize=7.5)
ax.set_title('Timescale recovery, per mode', fontsize=9)
ax.legend(handles=style_handles, fontsize=5.8, loc='upper left', **LEG)
P.panel_label(ax, 'e', dx=-0.16)

save(fig, 'fig_modes.svg')

## Fig. 4 — a rhythm added to a signal and a rhythm that is part of it (`fig_oscillations.svg`)

Panels a, b, d, e simulate and fit one example each in the cell; panels c and f
read the sweeps written by `scripts/run_03_oscillations.py`.

In [ ]:
# ---- Fig. 4: fig_oscillations ----------------------------------------------
sweep = pd.read_csv(RES / 'oscillation_sweep.csv.gz')
damped = pd.read_csv(RES / 'damped_grid.csv.gz')
SNRS = [0.125, 0.25, 0.5, 1.0, 2.0, 4.0]
DAMPED_TAUS = [0.02, 0.05, 0.15]
show = ['ar1_acf', 'arp_acf', 'ar1_psd', 'arp_psd']


def median_curve(df, method, column, xs, xcol, fixed=None):
    med = []
    for x in xs:
        sel = np.isclose(df[xcol], x) & (df['method'] == method)
        for key, val in (fixed or {}).items():
            sel &= np.isclose(df[key], val)
        v = df.loc[sel, column].to_numpy(dtype=float)
        v = v[np.isfinite(v) & (v > 0)]
        med.append(np.median(v) if len(v) else np.nan)
    return np.asarray(med)


rng = np.random.default_rng(3)
rows = [(AR1PlusSine(tau=TAU, f_osc=10.0, snr=1.0), 'AR(1) + Oscillation'),
        (AR2Damped(tau=TAU, f_osc=10.0), 'AR(2) oscillator')]
LEG = dict(frameon=True, framealpha=1.0, edgecolor='none', ncol=2,
           columnspacing=0.6, handlelength=1.4, fontsize=5.6)

fig = plt.figure(figsize=(7.6, 4.4))
gs = fig.add_gridspec(2, 3, hspace=0.70, wspace=0.42)
for r, (proc, title) in enumerate(rows):
    signal = prepare(proc.simulate(200.0, FS, rng), FS, welch_win_s=4.0)
    fit1 = estimate(signal, 'ar1_psd', robust=True)
    fitp = estimate(signal, 'arp_psd', robust=True)   # BIC picks the order

    # -- the signal ------------------------------------------------------
    ax = fig.add_subplot(gs[r, 0])
    t = np.arange(int(1.5 * FS)) / FS
    y = signal.sig[:len(t)]
    ax.plot(t, y / np.std(y), color='k', lw=0.6)
    ax.set_ylim(-4, 4)
    ax.set_yticks([])
    ax.set_xlabel('Time (s)', fontsize=9)
    ax.spines['left'].set_visible(False)
    ax.set_title(title, fontsize=10, loc='left')
    ax.tick_params(labelsize=8)
    P.panel_label(ax, 'ad'[r], dx=-0.16)

    # -- the spectrum ----------------------------------------------------
    ax = fig.add_subplot(gs[r, 1])
    m = signal.fit_mask
    f = signal.freqs[m]
    ax.loglog(f, signal.powers[m], color='k', lw=1.0, label='data')
    ax.loglog(f, ar_psd(fit1.phi, f, FS, fit1.offset), color='#E69F00',
              lw=1.6, ls=(0, (5, 2)), label='AR(1)')
    if fitp.order > 1:      # in row 1 the order search returns p = 1
        ax.loglog(f, ar_psd(fitp.phi, f, FS, fitp.offset), color='#0072B2',
                  lw=1.6, ls=(0, (1, 1.4)), label=f'AR({fitp.order})')
    ax.set_xlabel('Frequency (Hz)', fontsize=9)
    ax.set_ylabel('Power', fontsize=9)
    ax.set_yticks([])
    ax.legend(fontsize=7, loc='lower left')
    ax.tick_params(labelsize=8)
    P.panel_label(ax, 'be'[r], dx=-0.20)

    # -- what the estimators report ---------------------------------------
    ax = fig.add_subplot(gs[r, 2])
    if r == 0:
        for method in show:
            med = median_curve(sweep, method, 'tau_relaxation', SNRS,
                               'param_snr', {'param_f_osc': 10.0})
            ax.plot(SNRS, med, 'o-', ms=3, lw=1.2, color=P.METHOD_COLORS[method],
                    label=ESTIMATOR_LABELS[method].replace('AR(p)', 'AR($p$)'))
        ax.axhline(TAU, color='k', ls='--', lw=1.0)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('Oscillation / aperiodic power', fontsize=9)
        ax.set_ylabel('Aperiodic $\\hat\\tau$ (s)', fontsize=9)
        ax.set_title('Bias', fontsize=10)
        ax.set_ylim(8e-3, 2e-1)
        ax.legend(loc='upper left', **LEG)
    else:
        for method, lab in (('arp_acf', 'AR($p$) ACF'), ('arp_psd', 'AR($p$) PSD')):
            med = median_curve(damped, method, 'top_osc_tau', DAMPED_TAUS,
                               'param_tau', {'param_f_osc': 10.0})
            ax.plot(DAMPED_TAUS, med, 'o-', ms=4, lw=1.2,
                    color=P.METHOD_COLORS[method], label=lab)
        for method in ('ar1_acf', 'ar1_psd'):
            med = median_curve(damped, method, 'tau', DAMPED_TAUS,
                               'param_tau', {'param_f_osc': 10.0})
            ax.plot(DAMPED_TAUS, med, 's--', ms=3, lw=1.0,
                    color=P.METHOD_COLORS[method], label=ESTIMATOR_LABELS[method])
        lims = [0.012, 0.45]
        ax.plot(lims, lims, color='k', ls=':', lw=1.0)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('True envelope decay $\\tau_{env}$ (s)', fontsize=9)
        ax.set_ylabel('Estimated $\\tau_{env}$ (s)', fontsize=9)
        ax.set_title('Envelope decay', fontsize=10)
        ax.set_ylim(8e-3, 1.1)
        ax.legend(loc='upper left', **LEG)
    ax.tick_params(labelsize=8)
    P.panel_label(ax, 'cf'[r], dx=-0.26, dy=1.16)

save(fig, 'fig_oscillations.svg')

## Fig. 5 — how much data a timescale needs (`fig_finite.svg`)

Panels a and b reproduce the old `fig_2.svg`: the least-squares AR(1)
coefficient and its confidence interval against the true coefficient, for 100
and 1000 samples, averaged over 1000 simulations. Panel c is the sampling
distribution of $\hat\tau/\tau$, read from `results/finite_sampling.csv.gz`
(written by `scripts/run_04_finite.py`).

In [ ]:
# ---- Fig. 5: fig_finite ----------------------------------------------------
from scipy.signal import lfilter

rng = np.random.default_rng(5)
PHIS = np.linspace(0.5, 0.99, 25)
N_SIMS = 1000
BURN = 200


def phi_hat_ls(x):
    """Mean-corrected least-squares AR(1) coefficient for each row of x."""
    x = x - x.mean(axis=1, keepdims=True)
    return np.sum(x[:, 1:] * x[:, :-1], axis=1) / np.sum(x[:, :-1] ** 2, axis=1)


def ci_panel(ax, n, letter):
    mean_hat, lo, hi = [], [], []
    for phi in PHIS:
        eps = rng.normal(size=(N_SIMS, n + BURN))
        x = lfilter([1.0], [1.0, -phi], eps, axis=1)[:, BURN:]
        ph = phi_hat_ls(x)
        # Least-squares interval, eq. (var) of the paper, around each estimate.
        half = 1.96 * np.sqrt(np.maximum(1.0 - ph ** 2, 0.0) / (n - 1))
        mean_hat.append(ph.mean())
        lo.append((ph - half).mean())
        hi.append((ph + half).mean())
    ax.fill_between(PHIS, lo, hi, color=P.LIGHT, alpha=0.6, lw=0, label='95% CI')
    ax.plot(PHIS, PHIS, color='k', ls=':', lw=1.0, label='true')
    ax.plot(PHIS, mean_hat, color=FIT_COLOR, lw=1.5, label=r'mean $\hat\varphi$')
    ax.set_xlabel(r'True $\varphi$')
    ax.set_ylabel(r'Estimated $\varphi$')
    ax.set_title(f'n = {n} samples', fontsize=9)
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(0.3, 1.05)
    P.panel_label(ax, letter, dx=-0.28)


fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.4), gridspec_kw=dict(wspace=0.45))
ci_panel(axes[0], 100, 'a')
axes[0].legend(fontsize=6.5, loc='upper left', handlelength=1.6)
ci_panel(axes[1], 1000, 'b')

# (c) the sampling distribution of tau-hat at three recording lengths --------
sim = pd.read_csv(RES / 'finite_sampling.csv.gz')
durations = np.sort(sim['duration'].unique())
ax = axes[2]
for tot, colour in zip([20, 100, 1000], ['#D55E00', '#E69F00', '#0072B2']):
    near = durations[np.argmin(np.abs(durations - TAU * tot))]
    vals = sim[(sim['process'] == 'ar1') & (sim['method'] == 'ar1_direct')
               & (sim['duration'] == near)]['tau'].to_numpy(float) / TAU
    vals = vals[np.isfinite(vals)]
    ax.hist(vals, bins=40, range=(0, 3), density=True, histtype='step',
            color=colour, lw=1.2, label=rf'$T = {tot}\tau$')
ax.axvline(1.0, color='k', ls='--', lw=1.0)
ax.set_xlabel(r'$\hat\tau\,/\,\tau$')
ax.set_ylabel('Density')
ax.set_title('Sampling distribution', fontsize=9)
ax.legend(fontsize=6.5, loc='upper right')
P.panel_label(ax, 'c', dx=-0.28)

save(fig, 'fig_finite.svg')

## Fig. 6 — the single-timescale model in eight recordings (`fig_realdata.svg`)

Fits every trace of every recording at draw time (about a minute) so that the
mode components in panel b and the boxes in panel c come from the same fits.
Reads `results/real_data_fits.csv` (for the lag-axis span) and `data/*.npz`.

In [ ]:
# ---- Fig. 6: fig_realdata --------------------------------------------------
fits = pd.read_csv(RES / 'real_data_fits.csv')
keys = D.available()
cache = {k: D.load(k) for k in keys}
N_MODES_SHOWN = 3
MIN_TRACE_FRACTION = 0.5      # draw a mode's box only if this many traces reach it


def resolvable_modes(fit, signal):
    """The fit's modes, most important first, dropping those slower than the
    fitted band can resolve: a mode with tau > 1 / (2 pi f_lo) has its knee
    below the band, so its timescale is an extrapolation, not a measurement."""
    if fit.modes is None:
        return []
    tau_max = 1.0 / (2.0 * np.pi * signal.f_range[0])
    keep = [m for m in fit.modes.modes if m.tau <= tau_max]
    return sorted(keep, key=lambda m: -m.importance)


n = len(keys)
fig = plt.figure(figsize=(6.4, 1.15 * n + 0.9))
gs = fig.add_gridspec(n, 3, hspace=0.45, wspace=0.42)
med = lambda a: np.median(np.vstack(a), axis=0)

for i, key in enumerate(keys):
    data = cache[key]
    sub = fits[fits['dataset'] == key]
    signal = D.prepare_trace(data['X'][0], data)
    mask = signal.fit_mask & signal.exclude_mask
    first, last = i == 0, i == n - 1

    # Median across traces of the data, the fits, and the ranked modes.
    psd_data, acf_data, psd_fit, acf_fit = [], [], [], []
    mode_curves = [[] for _ in range(N_MODES_SHOWN)]
    mode_taus = [[] for _ in range(N_MODES_SHOWN)]
    nlags = len(signal.corrs) - 1
    n_traces = 0
    for trace in data['X']:
        sig = D.prepare_trace(trace, data)
        if len(sig.corrs) - 1 != nlags:
            continue
        n_traces += 1
        psd_data.append(sig.powers[mask])
        acf_data.append(sig.corrs)
        fp = estimate(sig, 'arp_psd', robust=True)
        if fp.success and fp.phi is not None:
            psd_fit.append(ar_psd(fp.phi, sig.freqs[mask], sig.fs, fp.offset))
        fa = estimate(sig, 'arp_acf', robust=True)
        if not (fa.success and fa.phi is not None):
            continue
        height = fa.params['height']
        acf_fit.append(height * ar_acf_fft(fa.phi, nlags) + fa.params['acf_offset'])
        comps = fa.modes.acf_components(nlags)
        index = {id(m): j for j, m in enumerate(fa.modes.modes)}
        for r, mode in enumerate(resolvable_modes(fa, sig)[:N_MODES_SHOWN]):
            mode_curves[r].append(height * comps[index[id(mode)]])
            mode_taus[r].append(mode.tau * 1e3)
    psd_data, acf_data = med(psd_data), med(acf_data)
    enough = lambda a: len(a) >= MIN_TRACE_FRACTION * max(n_traces, 1)

    # -- a: spectrum -------------------------------------------------------
    ax = fig.add_subplot(gs[i, 0])
    ax.loglog(signal.freqs[mask], psd_data, color=DATA_COLOR, lw=1.1, label='data', zorder=1)
    if psd_fit:
        ax.loglog(signal.freqs[mask], med(psd_fit), color=FIT_COLOR, lw=FIT_LW,
                  ls=FIT_STYLE, label='AR($p$) fit', zorder=3)
    ax.set_ylabel(data['panel_name'], fontsize=7, labelpad=3)
    ax.set_yticks([])
    ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.tick_params(labelsize=6.5)
    if first:
        ax.set_title('Power spectrum', fontsize=9)
        ax.legend(fontsize=5.8, loc='lower left', handlelength=1.8, frameon=True,
                  framealpha=1.0, edgecolor='none', borderpad=0.2, labelspacing=0.2)
        P.panel_label(ax, 'a', dx=-0.24, dy=1.40)
    if last:
        ax.set_xlabel('Frequency (Hz)', fontsize=7.5)

    # -- b: autocorrelation with its modes -----------------------------------
    ax = fig.add_subplot(gs[i, 1])
    lags = np.arange(nlags + 1) / signal.fs
    ax.plot(lags, acf_data, color=DATA_COLOR, lw=1.1, zorder=1)
    for r in range(N_MODES_SHOWN):
        if enough(mode_curves[r]):
            ax.plot(lags, med(mode_curves[r]), color=RANK_COLORS[r], lw=1.2, zorder=2)
    if acf_fit:
        ax.plot(lags, med(acf_fit), color=FIT_COLOR, lw=FIT_LW, ls=FIT_STYLE, zorder=3)
    ax.axhline(0, color=P.LIGHT, lw=0.6, zorder=0)
    taus = [t for t in (sub['tau'].median(), sub['tau_relaxation'].median())
            if np.isfinite(t) and t > 0]
    span = float(np.clip(max(6 * max(taus), 12 / signal.fs), 4 / signal.fs, lags[-1]))
    ax.set_xlim(0, span)
    lo = float(np.min(acf_data[lags <= span]))
    ax.set_ylim(min(-0.25, 1.15 * lo), 1.05)
    ax.set_yticks([0, 1])
    ax.tick_params(labelsize=6.5)
    if first:
        ax.set_title('Autocorrelation', fontsize=9)
        handles = ([proxy(color=DATA_COLOR, lw=1.1, label='data')]
                   + [proxy(color=RANK_COLORS[r], lw=1.2, label=f'mode {r + 1}')
                      for r in range(N_MODES_SHOWN)]
                   + [proxy(color=FIT_COLOR, lw=FIT_LW, ls=FIT_STYLE, label='AR($p$) fit')])
        ax.legend(handles=handles, fontsize=5.5, loc='upper right', handlelength=1.8,
                  frameon=True, framealpha=1.0, edgecolor='none', borderpad=0.2,
                  labelspacing=0.18)
        P.panel_label(ax, 'b', dx=-0.22, dy=1.40)
    if last:
        ax.set_xlabel('Lag (s)', fontsize=7.5)

    # -- c: the timescale of each mode ---------------------------------------
    ax = fig.add_subplot(gs[i, 2])
    drawn = []
    for r in range(N_MODES_SHOWN):
        vals = np.array([v for v in mode_taus[r] if np.isfinite(v) and v > 0])
        if not enough(vals):
            continue
        drawn.append(vals)
        colour = RANK_COLORS[r]
        bp = ax.boxplot([vals], positions=[r], widths=0.6, patch_artist=True,
                        showfliers=False, medianprops=dict(color='k', lw=1.1),
                        whiskerprops=dict(color=colour, lw=0.8),
                        capprops=dict(color=colour, lw=0.8))
        for box in bp['boxes']:
            box.set(facecolor=colour, alpha=0.55, lw=0.7, edgecolor=colour)
    ax.set_yscale('log')
    ax.set_xlim(-0.7, N_MODES_SHOWN - 0.3)
    ax.set_xticks(np.arange(N_MODES_SHOWN))
    ax.set_xticklabels([f'mode {r + 1}' for r in range(N_MODES_SHOWN)] if last else [],
                       fontsize=5.6, rotation=90)
    ax.set_ylabel('ms', fontsize=7, labelpad=1)
    ax.tick_params(labelsize=6.5)
    if drawn:
        decade_ticks(ax, np.concatenate(drawn))
    if first:
        ax.set_title('Mode timescales', fontsize=9)
        P.panel_label(ax, 'c', dx=-0.30, dy=1.40)
    if last:
        ax.text(0.5, -0.92, 'by importance', ha='center', va='top', fontsize=6.5,
                color=P.GREY, transform=ax.transAxes)

save(fig, 'fig_realdata.svg')

## Fig. S1 — six estimators on five simulated processes (`figS_benchmark.svg`)

Reads `results/benchmark_main.csv.gz` (written by `scripts/run_02_benchmark.py`);
the example spectra in row b are fit in the cell.

In [ ]:
# ---- Fig. S1: figS_benchmark -----------------------------------------------
DURATION, WELCH_WIN_S = 100.0, 4.0
PROCESSES = [AR1(tau=TAU), AR1PlusSine(tau=TAU, f_osc=10.0, snr=1.0),
             AR2Damped(tau=TAU, f_osc=10.0), ARpModes(), PowerLaw(d=0.45)]
SHORT_LABELS = ['AR(1)', 'AR(1) + sine', 'AR(2) oscillator', 'AR(8), 5 modes', 'Power law']
OSCILLATORY_TRUTH = {'ar2_damped'}
YLIM = (-3.1, 1.7)                      # log10 seconds

df = pd.read_csv(RES / 'benchmark_main.csv.gz')
# The timescale each estimator is asked for: tau for AR(1) fits, the envelope
# decay for the oscillator, the most important relaxation mode otherwise.
df['tau_target'] = np.where(
    df['method'].str.startswith('ar1_'), df['tau'],
    np.where(df['process'].isin(OSCILLATORY_TRUTH), df['top_osc_tau'], df['tau_relaxation'])
).astype(float)

methods = list(P.METHOD_ORDER)
groups = [('time', 'time'), ('acf', 'ACF'), ('psd', 'PSD')]
n = len(PROCESSES)

fig = plt.figure(figsize=(7.3, 5.2))
gs = fig.add_gridspec(3, n, hspace=0.45, wspace=0.3, height_ratios=[0.75, 1.0, 1.15])
for j, (proc, label) in enumerate(zip(PROCESSES, SHORT_LABELS)):
    rng = np.random.default_rng(3)
    signal = prepare(proc.simulate(DURATION, FS, rng), FS, welch_win_s=WELCH_WIN_S)
    fit1 = estimate(signal, 'ar1_psd', robust=proc.has_oscillation)
    fitp = estimate(signal, 'arp_psd', robust=proc.has_oscillation)

    # -- row a: the signal ---------------------------------------------------
    ax = fig.add_subplot(gs[0, j])
    t = np.arange(int(1.5 * FS)) / FS
    y = signal.sig[:len(t)]
    ax.plot(t, y / np.std(y), color='k', lw=0.5)
    ax.set_ylim(-4, 4); ax.set_xticks([0, 1]); ax.set_yticks([])
    ax.tick_params(labelsize=7)
    ax.set_title(label, fontsize=8.5)
    ax.spines['left'].set_visible(False)
    if j == 0:
        ax.set_ylabel('signal', fontsize=8)
        P.panel_label(ax, 'a', dx=-0.30, dy=1.30)
    if j == n // 2:
        ax.set_xlabel('Time (s)', fontsize=8)

    # -- row b: spectrum and fits --------------------------------------------
    ax = fig.add_subplot(gs[1, j])
    m = signal.fit_mask
    f = signal.freqs[m]
    ax.loglog(f, signal.powers[m], color=P.LIGHT, lw=0.9)
    ax.loglog(f, ar_psd(fit1.phi, f, FS, fit1.offset), color=P.METHOD_COLORS['ar1_psd'],
              lw=1.4, ls='--', label='AR(1) fit')
    ax.loglog(f, ar_psd(fitp.phi, f, FS, fitp.offset), color=P.METHOD_COLORS['arp_psd'],
              lw=1.4, ls='-.', label='AR($p$) fit')
    ax.set_xticks([1, 100]); ax.set_yticks([])
    ax.tick_params(labelsize=7)
    if j == 0:
        ax.set_ylabel('Power', fontsize=8)
        ax.legend(fontsize=6.5, loc='lower left')
        P.panel_label(ax, 'b', dx=-0.30, dy=1.18)
    if j == n // 2:
        ax.set_xlabel('Frequency (Hz)', fontsize=8)

    # -- row c: what each estimator reports ----------------------------------
    ax = fig.add_subplot(gs[2, j])
    sub = df[df['process'] == proc.name]
    for i, method in enumerate(methods):
        vals = sub.loc[sub['method'] == method, 'tau_target'].to_numpy(float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        x = i + 0.9 * (i // 2)
        if not len(vals):
            ax.text(x, -2.9, 'n/a', fontsize=5.5, rotation=90, ha='center', color=P.GREY)
            continue
        colour = P.METHOD_COLORS[method]
        bp = ax.boxplot([np.log10(vals)], positions=[x], widths=0.62, patch_artist=True,
                        showfliers=False, medianprops=dict(color='k', lw=1.1),
                        whiskerprops=dict(color=colour, lw=0.8), capprops=dict(color=colour, lw=0.8))
        for box in bp['boxes']:
            box.set(facecolor=colour, alpha=0.6, lw=0.7, edgecolor=colour)
        if np.log10(np.median(vals)) > YLIM[1]:
            ax.plot([x], [YLIM[1] - 0.12], marker='^', ms=5, clip_on=False, color=colour, mec='none')
    if np.isfinite(proc.tau_true):
        colour = P.MODE_COLORS['oscillatory'] if proc.name in OSCILLATORY_TRUTH else 'k'
        ax.axhline(np.log10(proc.tau_true), color=colour, ls='--', lw=1.0)
    tint = proc.tau_integrated(FS)
    if np.isfinite(tint):
        ax.axhline(np.log10(tint), color='#FFB000', ls=':', lw=1.3)
    ax.set_xlim(-0.8, 7.6); ax.set_ylim(*YLIM)
    ax.set_xticks([0, 1, 2.9, 3.9, 5.8, 6.8])
    ax.set_xticklabels(['AR(1)', 'AR($p$)'] * 3, fontsize=5.6, rotation=90)
    for x, (domain, name) in zip([0.5, 3.4, 6.3], groups):
        ax.text(x, -0.46, name, ha='center', va='top', fontsize=7.5,
                color=P.DOMAIN_COLORS[domain], transform=ax.get_xaxis_transform())
    P.log_tick_labels(ax)
    ax.tick_params(labelsize=7)
    if j == 0:
        ax.set_ylabel('Estimated $\\tau$ (s)', fontsize=8)
        P.panel_label(ax, 'c', dx=-0.30, dy=1.14)
    else:
        ax.set_yticklabels([])

handles = [proxy(color='k', ls='--', lw=1.0, label='true $\\tau$'),
           proxy(color=P.MODE_COLORS['oscillatory'], ls='--', lw=1.0,
                 label='envelope decay of the oscillator'),
           proxy(color='#FFB000', ls=':', lw=1.3, label='integrated $\\tau$ (area under the ACF)')]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=7.0,
           bbox_to_anchor=(0.5, -0.12), frameon=False)

save(fig, 'figS_benchmark.svg')

## Fig. S2 — the full oscillation sweep (`figS_oscillation_grid.svg`)

Reads `results/oscillation_sweep.csv.gz` (written by `scripts/run_03_oscillations.py`).

In [ ]:
# ---- Fig. S2: figS_oscillation_grid ----------------------------------------
sweep = pd.read_csv(RES / 'oscillation_sweep.csv.gz')
SNRS = [0.125, 0.25, 0.5, 1.0, 2.0, 4.0]
FREQS = [4.0, 6.0, 10.0, 16.0, 25.0, 40.0]
show = ['ar1_acf', 'arp_acf', 'ar1_psd', 'arp_psd']

fig, axes = plt.subplots(1, len(show), figsize=(7.4, 2.1))
for ax, method in zip(axes, show):
    grid = np.full((len(SNRS), len(FREQS)), np.nan)
    for i, snr in enumerate(SNRS):
        for j, f_osc in enumerate(FREQS):
            sel = ((sweep['method'] == method) & np.isclose(sweep['param_snr'], snr)
                   & np.isclose(sweep['param_f_osc'], f_osc))
            vals = sweep.loc[sel, 'tau_relaxation'].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals):
                grid[i, j] = np.median(vals) / TAU
    # Colour and number are the same quantity: estimate / truth, log-spaced so
    # a factor of two too big and too small are equally far from white.
    im = ax.imshow(np.log10(grid), origin='lower', cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    for i in range(len(SNRS)):
        for j in range(len(FREQS)):
            value = grid[i, j]
            txt = f'{value:.2f}' if np.isfinite(value) else 'none'
            ax.text(j, i, txt, ha='center', va='center', fontsize=4.4,
                    color='k' if abs(np.log10(max(value, 1e-9))) < 0.6 else 'white')
    ax.set_xticks(range(len(FREQS)))
    ax.set_xticklabels([f'{f:g}' for f in FREQS], fontsize=6)
    ax.set_yticks(range(len(SNRS)))
    ax.set_yticklabels([f'{v:g}' for v in SNRS] if ax is axes[0] else [], fontsize=6)
    ax.set_xlabel('Oscillation freq. (Hz)', fontsize=7)
    ax.set_title(ESTIMATOR_LABELS[method].replace('AR(p)', 'AR($p$)'), fontsize=8)
axes[0].set_ylabel('Oscillation /\naperiodic power', fontsize=7)
ticks = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0]
cb = fig.colorbar(im, ax=axes, fraction=0.03, pad=0.02, ticks=np.log10(ticks))
cb.ax.set_yticklabels([f'{t:g}' for t in ticks], fontsize=6)
cb.set_label('estimated $\\tau$ / true $\\tau$', fontsize=7)

save(fig, 'figS_oscillation_grid.svg')

## Fig. S3 — real recordings cut into windows (`figS_windows.svg`)

Reads `results/empirical_truncation_summary.csv` (written by
`scripts/run_06_empirical_truncation.py`). These were panels e and f of the
old `fig_windows.svg`; the simulation panels of that figure are now Fig. 5.

In [ ]:
# ---- Fig. S3: figS_windows -------------------------------------------------
emp = pd.read_csv(RES / 'empirical_truncation_summary.csv')
keys = [k for k in D.DATASET_KEYS if k in set(emp['dataset'])]
colours = dict(zip(keys, plt.cm.viridis(np.linspace(0, 0.9, len(keys)))))

fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.7), gridspec_kw=dict(wspace=0.55))

# (a) the median estimate against window length -------------------------------
ax = axes[0]
for key in keys:
    sub = emp[(emp['dataset'] == key) & (emp['method'] == 'ar1_acf')]
    ax.plot(sub['duration'] / sub['reference_tau'], sub['rel_to_longest'], 'o-',
            ms=2.5, lw=1.0, color=colours[key], label=D.SHORT_NAMES.get(key, key))
ax.axhline(1.0, color='k', ls=':', lw=0.9)
ax.axhspan(0.8, 1.25, color=P.LIGHT, alpha=0.35, lw=0)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.8, 3e4); ax.set_ylim(0.2, 3.2)
ax.set_yticks([0.25, 0.5, 1, 2])
ax.set_yticklabels(['0.25', '0.5', '1', '2'])
ax.set_xlabel(r'Window length / $\hat\tau$')
ax.set_ylabel(r'$\hat\tau(T)\,/\,\hat\tau$(longest)')
ax.set_title('Window dependence', fontsize=9)
ax.legend(fontsize=5.2, ncol=2, loc='lower right', columnspacing=0.6,
          handlelength=1.1, frameon=True, framealpha=1.0, edgecolor='none')
P.panel_label(ax, 'a', dx=-0.22)

# (b) the spread between windows, against the sampling bound ------------------
ax = axes[1]
height = 0.36
for i, method in enumerate(('ar1_acf', 'ar1_psd')):
    vals = [emp[(emp['dataset'] == k) & (emp['method'] == method)]['excess_sd'].iloc[-1]
            for k in keys]
    ax.barh(np.arange(len(keys)) + (i - 0.5) * height, vals, height=height,
            color=P.METHOD_COLORS[method], label=ESTIMATOR_LABELS[method])
ax.axvline(1.0, color='k', ls='--', lw=1.0)
ax.set_xscale('log')
ax.set_yticks(np.arange(len(keys)))
ax.set_yticklabels([D.SHORT_NAMES.get(k, k) for k in keys], fontsize=6)
ax.set_xlabel('Observed spread / sampling bound')
ax.set_title('Beyond sampling noise', fontsize=9)
ax.legend(fontsize=6, loc='lower right', handlelength=1.2, borderpad=0.25,
          frameon=True, framealpha=1.0, edgecolor='none')
P.panel_label(ax, 'b', dx=-0.42)

save(fig, 'figS_windows.svg')